In [ ]:
# ============================================
# CELL 1: Setup for EDA
# ============================================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import os

# Create visuals folder if not exists
os.makedirs('../visuals', exist_ok=True)

# Set style
sns.set_theme(style="whitegrid", palette="husl")
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 12

# Load cleaned data
df = pd.read_csv('../data/Amazon_cleaned.csv')
df['date'] = pd.to_datetime(df['date'])

print("✅ Setup complete!")
print(f"📊 Loaded {len(df):,} clean records")

In [ ]:
# ============================================
# CHART 1: Monthly Revenue Trend
# ============================================
# Business Question: Which months generate most revenue?
# Why important: Helps plan inventory & marketing budget

monthly_revenue = (df.groupby(['year','month','month_name'])['revenue']
                   .sum()
                   .reset_index()
                   .sort_values(['year','month']))

fig, ax = plt.subplots(figsize=(14, 6))

ax.plot(range(len(monthly_revenue)), 
        monthly_revenue['revenue'], 
        marker='o', 
        linewidth=2.5, 
        color='#FF9900',  # Amazon orange!
        markersize=8,
        markerfacecolor='white',
        markeredgewidth=2)

# Add value labels on each point
for i, row in monthly_revenue.iterrows():
    ax.annotate(f"₹{row['revenue']/1000:.0f}K",
                xy=(list(monthly_revenue.index).index(i), row['revenue']),
                textcoords="offset points",
                xytext=(0, 10),
                ha='center', fontsize=9)

ax.set_xticks(range(len(monthly_revenue)))
ax.set_xticklabels(monthly_revenue['month_name'], rotation=45, ha='right')
ax.set_title('📈 Monthly Revenue Trend — Amazon Sales', 
             fontsize=16, fontweight='bold', pad=20)
ax.set_xlabel('Month', fontsize=12)
ax.set_ylabel('Total Revenue (₹)', fontsize=12)
ax.yaxis.set_major_formatter(
    mticker.FuncFormatter(lambda x, p: f'₹{x:,.0f}'))

plt.tight_layout()
plt.savefig('../visuals/01_monthly_revenue_trend.png', 
            dpi=150, bbox_inches='tight')
plt.show()
print("✅ Chart saved to visuals/01_monthly_revenue_trend.png")

In [ ]:
# ============================================
# CHART 2: Revenue by Category
# ============================================
# Business Question: Which product categories 
# drive the most revenue?
# Why important: Focus marketing on top categories

category_revenue = (df.groupby('category')['revenue']
                    .sum()
                    .sort_values(ascending=True)
                    .tail(10))

fig, ax = plt.subplots(figsize=(12, 7))

bars = ax.barh(category_revenue.index, 
               category_revenue.values,
               color=sns.color_palette("YlOrRd", len(category_revenue)),
               edgecolor='white',
               height=0.6)

# Add value labels
for bar, val in zip(bars, category_revenue.values):
    ax.text(val + (category_revenue.max() * 0.01),
            bar.get_y() + bar.get_height()/2,
            f'₹{val:,.0f}',
            va='center', fontsize=10, fontweight='bold')

ax.set_title('🏆 Top 10 Categories by Revenue', 
             fontsize=16, fontweight='bold', pad=20)
ax.set_xlabel('Total Revenue (₹)', fontsize=12)
ax.set_ylabel('Category', fontsize=12)
ax.xaxis.set_major_formatter(
    mticker.FuncFormatter(lambda x, p: f'₹{x:,.0f}'))

plt.tight_layout()
plt.savefig('../visuals/02_top_categories_revenue.png', 
            dpi=150, bbox_inches='tight')
plt.show()
print("✅ Chart saved to visuals/02_top_categories_revenue.png")

In [ ]:
# ============================================
# CHART 3: Order Status Distribution
# ============================================
# Business Question: What % of orders are 
# delivered, cancelled, returned?
# Why important: High cancellation = problem!

status_counts = df['status'].value_counts()

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

# Pie Chart
colors = ['#2ecc71','#e74c3c','#f39c12','#3498db','#9b59b6','#1abc9c']
wedges, texts, autotexts = ax1.pie(
    status_counts.values,
    labels=status_counts.index,
    autopct='%1.1f%%',
    colors=colors[:len(status_counts)],
    startangle=90,
    pctdistance=0.85)

for text in autotexts:
    text.set_fontsize(10)
    text.set_fontweight('bold')

ax1.set_title('Order Status Distribution', 
              fontsize=14, fontweight='bold')

# Bar Chart
bars = ax2.bar(status_counts.index, 
               status_counts.values,
               color=colors[:len(status_counts)],
               edgecolor='white')

for bar, val in zip(bars, status_counts.values):
    ax2.text(bar.get_x() + bar.get_width()/2,
             bar.get_height() + 50,
             f'{val:,}',
             ha='center', fontsize=10, fontweight='bold')

ax2.set_title('Order Count by Status', 
              fontsize=14, fontweight='bold')
ax2.set_xlabel('Status', fontsize=11)
ax2.set_ylabel('Number of Orders', fontsize=11)
ax2.tick_params(axis='x', rotation=45)

plt.suptitle('📦 Order Status Analysis', 
             fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('../visuals/03_order_status.png', 
            dpi=150, bbox_inches='tight')
plt.show()
print("✅ Chart saved to visuals/03_order_status.png")

In [ ]:
# ============================================
# CHART 4: Top 10 States by Revenue
# ============================================
# Business Question: Which states generate 
# most revenue?
# Why important: Helps decide warehouse locations

state_revenue = (df.groupby('ship_state')['revenue']
                 .sum()
                 .sort_values(ascending=False)
                 .head(10))

fig, ax = plt.subplots(figsize=(13, 6))

bars = ax.bar(state_revenue.index, 
              state_revenue.values,
              color=sns.color_palette("Blues_r", len(state_revenue)),
              edgecolor='white')

for bar, val in zip(bars, state_revenue.values):
    ax.text(bar.get_x() + bar.get_width()/2,
            bar.get_height() + (state_revenue.max() * 0.01),
            f'₹{val/100000:.1f}L',
            ha='center', fontsize=9, fontweight='bold')

ax.set_title('🗺️ Top 10 States by Revenue', 
             fontsize=16, fontweight='bold', pad=20)
ax.set_xlabel('State', fontsize=12)
ax.set_ylabel('Total Revenue (₹)', fontsize=12)
ax.tick_params(axis='x', rotation=45)
ax.yaxis.set_major_formatter(
    mticker.FuncFormatter(lambda x, p: f'₹{x:,.0f}'))

plt.tight_layout()
plt.savefig('../visuals/04_top_states_revenue.png', 
            dpi=150, bbox_inches='tight')
plt.show()
print("✅ Chart saved to visuals/04_top_states_revenue.png")

In [ ]:
# ============================================
# CHART 5: Sales Channel Performance
# ============================================
# Business Question: Amazon.in vs Easy Ship 
# which channel performs better?

channel_data = df.groupby('sales_channel').agg(
    Total_Revenue=('revenue', 'sum'),
    Total_Orders=('revenue', 'count'),
    Avg_Order_Value=('amount', 'mean')
).reset_index()

fig, axes = plt.subplots(1, 3, figsize=(16, 6))

metrics = ['Total_Revenue', 'Total_Orders', 'Avg_Order_Value']
titles = ['Total Revenue', 'Total Orders', 'Avg Order Value']
colors = ['#FF9900', '#146EB4', '#2ecc71']

for i, (metric, title, color) in enumerate(
        zip(metrics, titles, colors)):
    bars = axes[i].bar(channel_data['sales_channel'],
                       channel_data[metric],
                       color=color,
                       alpha=0.85,
                       edgecolor='white')
    
    for bar, val in zip(bars, channel_data[metric]):
        axes[i].text(bar.get_x() + bar.get_width()/2,
                     bar.get_height() * 1.02,
                     f'{val:,.0f}',
                     ha='center', fontsize=9,
                     fontweight='bold')
    
    axes[i].set_title(title, fontsize=13, fontweight='bold')
    axes[i].set_xlabel('Sales Channel', fontsize=10)
    axes[i].tick_params(axis='x', rotation=30)

plt.suptitle('🛒 Sales Channel Performance Analysis',
             fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig('../visuals/05_sales_channel.png', 
            dpi=150, bbox_inches='tight')
plt.show()
print("✅ Chart saved to visuals/05_sales_channel.png")

In [ ]:
# ============================================
# CHART 6: Order Size Distribution
# ============================================
# Business Question: Are most orders small, 
# medium or large value?

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

# Distribution of order amounts
ax1.hist(df['amount'], bins=50, 
         color='#FF9900', edgecolor='white', alpha=0.8)
ax1.axvline(df['amount'].mean(), color='red', 
            linestyle='--', linewidth=2,
            label=f"Mean: ₹{df['amount'].mean():,.0f}")
ax1.axvline(df['amount'].median(), color='blue',
            linestyle='--', linewidth=2,
            label=f"Median: ₹{df['amount'].median():,.0f}")
ax1.set_title('Distribution of Order Amounts',
              fontsize=13, fontweight='bold')
ax1.set_xlabel('Order Amount (₹)', fontsize=11)
ax1.set_ylabel('Frequency', fontsize=11)
ax1.legend()

# Order size category counts
size_counts = df['order_size'].value_counts()
colors_size = ['#2ecc71', '#f39c12', '#e74c3c']
bars = ax2.bar(size_counts.index, size_counts.values,
               color=colors_size, edgecolor='white')

for bar, val in zip(bars, size_counts.values):
    ax2.text(bar.get_x() + bar.get_width()/2,
             bar.get_height() + 20,
             f'{val:,}\n({val/len(df)*100:.1f}%)',
             ha='center', fontsize=10, fontweight='bold')

ax2.set_title('Orders by Size Category',
              fontsize=13, fontweight='bold')
ax2.set_xlabel('Order Size', fontsize=11)
ax2.set_ylabel('Number of Orders', fontsize=11)

plt.suptitle('💰 Order Value Analysis',
             fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig('../visuals/06_order_size_distribution.png',
            dpi=150, bbox_inches='tight')
plt.show()
print("✅ Chart saved to visuals/06_order_size_distribution.png")

In [ ]:
# ============================================
# CHART 7: Sales by Day of Week
# ============================================
# Business Question: Which day gets most orders?
# Why important: Schedule ads on peak days!

day_order = ['Monday','Tuesday','Wednesday',
             'Thursday','Friday','Saturday','Sunday']

day_data = (df.groupby('day_of_week')
            .agg(Orders=('revenue','count'),
                 Revenue=('revenue','sum'))
            .reindex(day_order))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

# Orders by day
colors_day = ['#e74c3c' if d in ['Saturday','Sunday'] 
              else '#3498db' for d in day_order]

bars = ax1.bar(day_data.index, day_data['Orders'],
               color=colors_day, edgecolor='white')
ax1.set_title('Orders per Day of Week',
              fontsize=13, fontweight='bold')
ax1.set_xlabel('Day', fontsize=11)
ax1.set_ylabel('Number of Orders', fontsize=11)
ax1.tick_params(axis='x', rotation=45)

# Revenue by day
ax2.bar(day_data.index, day_data['Revenue'],
        color=colors_day, edgecolor='white')
ax2.set_title('Revenue per Day of Week',
              fontsize=13, fontweight='bold')
ax2.set_xlabel('Day', fontsize=11)
ax2.set_ylabel('Total Revenue (₹)', fontsize=11)
ax2.tick_params(axis='x', rotation=45)
ax2.yaxis.set_major_formatter(
    mticker.FuncFormatter(lambda x, p: f'₹{x:,.0f}'))

plt.suptitle('📅 Day of Week Sales Pattern\n'
             '(Red = Weekend, Blue = Weekday)',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../visuals/07_day_of_week_sales.png',
            dpi=150, bbox_inches='tight')
plt.show()
print("✅ Chart saved to visuals/07_day_of_week_sales.png")

In [ ]:
# ============================================
# CHART 8: Quarterly Revenue Performance
# ============================================
# Business Question: Which quarter is strongest?
# Why important: Quarterly business planning

quarterly = df.groupby('quarter').agg(
    Revenue=('revenue', 'sum'),
    Orders=('revenue', 'count')
).reset_index()

quarterly['quarter_label'] = 'Q' + quarterly['quarter'].astype(str)

fig, ax = plt.subplots(figsize=(10, 6))

bars = ax.bar(quarterly['quarter_label'],
              quarterly['Revenue'],
              color=['#3498db','#2ecc71','#f39c12','#e74c3c'],
              edgecolor='white',
              width=0.5)

for bar, row in zip(bars, quarterly.itertuples()):
    ax.text(bar.get_x() + bar.get_width()/2,
            bar.get_height() + (quarterly['Revenue'].max()*0.01),
            f'₹{row.Revenue/100000:.1f}L\n({row.Orders:,} orders)',
            ha='center', fontsize=10, fontweight='bold')

ax.set_title('📊 Quarterly Revenue Performance',
             fontsize=16, fontweight='bold', pad=20)
ax.set_xlabel('Quarter', fontsize=12)
ax.set_ylabel('Total Revenue (₹)', fontsize=12)
ax.yaxis.set_major_formatter(
    mticker.FuncFormatter(lambda x, p: f'₹{x:,.0f}'))

plt.tight_layout()
plt.savefig('../visuals/08_quarterly_performance.png',
            dpi=150, bbox_inches='tight')
plt.show()
print("✅ Chart saved to visuals/08_quarterly_performance.png")